# Bible Chatbot - English Version

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
astra_db_endpoint = os.getenv("ASTRADB_ENDPOINT")
astra_db_token = os.getenv("ASTRADB_APPLICATION_TOKEN")
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

### Loading Data

In [2]:
from langchain_community.document_loaders import PyPDFDirectoryLoader

data_path = "../data/English/"

loader = PyPDFDirectoryLoader(data_path)

docs = loader.load()

print(f"Loaded {len(docs)} documents")

Loaded 1871 documents


In [3]:
docs[0]

Document(metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'Microsoft® Word 2013', 'creationdate': '2018-06-28T21:57:28-07:00', 'title': 'Holy Bible - New International Version', 'author': 'Biiible', 'moddate': '2018-06-28T14:14:01+00:00', 'source': '..\\data\\English\\whole_bible_niv1984.pdf', 'total_pages': 1871, 'page': 0, 'page_label': '1'}, page_content='1 \n \n  \nHoly Bible \nNew International Version \n \n \n \n \n \n \n \n \nAbout the New International Version –  \nThe New International Version was undertaken by an independent \ncommittee in after a general consensus that there was a need for a \nnew, contemporary English translation of the Bible. \n \nWith the help of scholars from all over the world, and multiple reviews \nfrom a committee of multiple denominations, the New International \nVersion has earned the widespread respect of all Christians as one of \nthe best translations available.')

### Splitting the data into chunks

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=2000, chunk_overlap=200, length_function=len
)

splits = splitter.split_documents(docs)

print(f"Split into {len(splits)} chunks")

Split into 3398 chunks


### Creating Vector Store

In [5]:
from astrapy.info import VectorServiceOptions
from langchain_astradb import AstraDBVectorStore

hf_vectorize_options = VectorServiceOptions(
            provider="huggingface",
            model_name="sentence-transformers/all-MiniLM-L6-v2",
            authentication={
                "providerKey": "bible_embeddings",
            },
        )


vector_store = AstraDBVectorStore(
    collection_name="english_bible",
    api_endpoint=astra_db_endpoint,
    token=astra_db_token,
    collection_vector_service_options=hf_vectorize_options
)

In [6]:
# create a function for encoding id to duplicate prevention
import hashlib
from langchain_core.documents import Document


def generate_doc_id(doc: Document) -> str:
    # Create a unique identifier based on the content of the document
    content = doc.page_content
    # Use a hash function to generate a unique identifier
    doc_id = hashlib.sha256(content.encode()).hexdigest()
    return doc_id

In [7]:
## RUN only once
# upload documents by preventing duplicates
import time

start = time.time()
ids = [generate_doc_id(doc) for doc in splits]

vector_store.add_documents(splits, ids=ids)

print(f'embedded and uploaded {len(splits)} in {time.time() - start} seconds')

embedded and uploaded 3398 in 7.204879522323608 seconds


### Querying the vector store

In [8]:
sim_docs = vector_store.similarity_search("In the beginning, God created")
sim_docs

[Document(id='5483614b8e3ba44e8ce36c7a766c669e8b6e3a4b17120a7cd1a7137a193c2055', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'Microsoft® Word 2013', 'creationdate': '2018-06-28T21:57:28-07:00', 'title': 'Holy Bible - New International Version', 'author': 'Biiible', 'moddate': '2018-06-28T14:14:01+00:00', 'source': '..\\data\\English\\whole_bible_niv1984.pdf', 'total_pages': 1871, 'page': 4, 'page_label': '5'}, page_content='5 \n \nWhen the Lord God made the earth and \nthe heavens-  \n5and no shrub of the field had y et \nappeared on the earth and no plant of \nthe field had yet sprung up, for the Lord \nGod had not sent rain on the earth and \nthere was no man to work the ground,  \n6but streams came up from the earth \nand watered the whole surface of the \nground-  \n7the Lord God formed the man from the \ndust of the ground and breathed into his \nnostrils the breath of life, and the man \nbecame a living being.  \n8N

### Creating llm and Prompt Template

In [9]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        (
            "You are a helpful assistant that responds to user queries.\n"
            "Use the provided context to answer the user's question.\n"
            "Do not make up or hallucinate any information.\n\n"
            "Context:\n{context}"
        ),
    ),
    ("human", "{input}"),
])



In [10]:
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain.chains.retrieval import create_retrieval_chain
from langchain_groq import ChatGroq

llm = ChatGroq(temperature=0, model_name="llama-3.3-70b-versatile")
docs_chain = create_stuff_documents_chain(
    llm=llm,
    prompt=prompt
)
retriever = vector_store.as_retriever()

chain = create_retrieval_chain(retriever, docs_chain)

In [11]:
query = "get me the reference of psalms 1 1?"
response = chain.invoke({"input": query})

In [12]:
response['answer']

'The reference for Psalms 1:1 is:\n\n"Blessed is the man who does not walk in the counsel of the wicked or stand in the way of sinners or sit in the seat of mockers."'

In [13]:
response['context']

[Document(id='f761f0179b8de5259943a547e168bdb24db710e59e56c205036bbf9dd299a4b0', metadata={'producer': '3-Heights(TM) PDF Optimization Shell 4.8.25.2 (http://www.pdf-tools.com)', 'creator': 'Microsoft® Word 2013', 'creationdate': '2018-06-28T21:57:28-07:00', 'title': 'Holy Bible - New International Version', 'author': 'Biiible', 'moddate': '2018-06-28T14:14:01+00:00', 'source': '..\\data\\English\\whole_bible_niv1984.pdf', 'total_pages': 1871, 'page': 857, 'page_label': '858'}, page_content='858 \n \nPsalms \nPSALM 1 \n1Blessed is the man who does not walk \nin the counsel of the wicked or stand in \nthe way of sinners or sit in the seat of \nmockers.  \n2But his delight is in the law of the Lord, \nand on his law he meditates day and \nnight.  \n3He is like a tree  planted by streams of \nwater, which yields its fruit in season \nand whose leaf does not wither. \nWhatever he does prospers.  \n4Not so the wicked! They are like chaff \nthat the wind blows away.  \n5Therefore the wicked 